# 04g - Master Forecast Showcase

This notebook generates final combined visualizations and business-oriented forecasts:
- **Cumulative Revenue Growth**: Predicting total volume over the next 30 days.
- **Weekly Hot-Spot Matrix**: Identifying strongest/weakest days of the week.
- **Final Model Ensembling**: Comparing Accuracy (ARIMA) vs Seasonality (Prophet).

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
cwd = Path.cwd()
ARTIFACTS_DIR = cwd if cwd.name == 'artifacts' else (cwd/ 'walmart' / 'artifacts' if (cwd/'walmart').exists() else (cwd.parent / 'artifacts'))

df = pd.read_csv(ARTIFACTS_DIR / 'daily_series.csv', parse_dates=['date']).set_index('date')
final_forecasts = pd.read_csv(ARTIFACTS_DIR / 'final_forecasts.csv', parse_dates=['date']).set_index('date')
print("Master Showcase Ready.")

In [ ]:
def plot_cumulative_projection():
    rev_forecast = final_forecasts['arima_daily_revenue'].cumsum()
    history_recent = df['daily_revenue'].tail(30).cumsum()
    
    plt.figure(figsize=(12, 6))
    plt.plot(rev_forecast, label='Projected 30-Day Cumulative Revenue', marker='o', color='forestgreen')
    plt.title('Cumulative Revenue Projection (Next 30 Days)')
    plt.ylabel('Incremental Total Revenue')
    plt.fill_between(rev_forecast.index, 0, rev_forecast, color='green', alpha=0.1)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(ARTIFACTS_DIR / '04g_cumulative_revenue_projection.png')
    plt.show()

plot_cumulative_projection()

In [ ]:
def plot_weekly_heatmaps():
    df['day'] = df.index.day_name()
    df['month'] = df.index.month_name()
    
    weekly_avg = df.groupby('day')[['daily_revenue', 'daily_orders']].mean()
    # Sort days correctly
    days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    weekly_avg = weekly_avg.reindex(days)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    sns.heatmap(weekly_avg[['daily_revenue']].T, annot=True, fmt=".0f", cmap="YlOrRd", ax=ax1)
    ax1.set_title('Average Revenue by Day of Week')
    sns.heatmap(weekly_avg[['daily_orders']].T, annot=True, fmt=".1f", cmap="YlGnBu", ax=ax2)
    ax2.set_title('Average Orders by Day of Week')
    
    plt.tight_layout()
    plt.savefig(ARTIFACTS_DIR / '04g_weekly_behavior_heatmap.png')
    plt.show()

plot_weekly_heatmaps()